# Data Preprocessing for Kaggle Dataset

## Importing libraries

In [32]:
# ! pip install pandas openpyxl emoji nltk datasets textblob

In [33]:
import re
import unicodedata
import numpy as np
import pandas as pd
import emoji
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import word_tokenize, pos_tag
from datasets import load_dataset
from textblob import TextBlob

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

True

## Dataset ingestion

In [34]:
path = "../../data/raw-data/corpus_kaggle.csv"
df = pd.read_csv(path, encoding="utf-8")

df["text"] = df["review_text"].fillna("").astype(str)

print(f"Loaded {path} | rows: {len(df)}\n")
print(df["sentiment"].value_counts(dropna=False))

Loaded ../../data/raw-data/corpus_kaggle.csv | rows: 50000

sentiment
Positive    27540
Neutral     12549
Negative     9911
Name: count, dtype: int64


In [35]:
texts = df["text"].astype(str)

print(f"Rows: {len(df)}")
print(f"Non-empty text: {texts.str.strip().str.len().gt(0).sum()}")
print(f"Avg chars: {texts.str.len().mean():.0f}")
print(f"Avg words: {texts.str.split().str.len().mean():.0f}")


Rows: 50000
Non-empty text: 50000
Avg chars: 63
Avg words: 11


In [36]:
emoji_counter = {}
for s in texts:
    for item in emoji.emoji_list(s):
        ch = item["emoji"]
        emoji_counter[ch] = emoji_counter.get(ch, 0) + 1
        
if not emoji_counter:
    print("No unicode emojis found in dataset.")
else:
    emoji_df = pd.DataFrame(
        sorted(emoji_counter.items(), key=lambda kv: kv[1], reverse=True),
        columns=["emoji", "count"],
    )
    print(f"Unique emojis: {len(emoji_df)}")
    emoji_df["demojize"] = emoji_df["emoji"].map(lambda e: emoji.demojize(e))
    display(emoji_df.head(20))

No unicode emojis found in dataset.


## Preprocessing functions
- For mircotext normalization, we use light slang expansion only by setting strict ALLOWLIST in _load_abbrev_patterns.
- Without creating an ALLOWLIST, Hugging Face maps texts like "so" --> "significant other".

In [37]:
# Remove HN quote markers ('>') while retaining usage in expressions e.g., A>B or A>=B
def remove_hn_blockquotes(text):
    lines = [line.lstrip(">").strip() for line in text.splitlines() if line.strip()]
    t = " ".join(lines)
    t = re.sub(r"\s>\s", " ", t)
    return re.sub(r"\s+", " ", t).strip()

# Remove markdown bold (**), italic (*), inline code (`), links [text](url)
def remove_markdown_formatting(text):
    t = text
    t = re.sub(r"\*\*(.+?)\*\*", r"\1", t)
    t = re.sub(r"\*(.+?)\*", r"\1", t)
    t = re.sub(r"`(.+?)`", r"\1", t)
    t = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", t)
    return t

# Expand contractions
CONTRACTIONS = {
    "don't": "do not", "doesn't": "does not", "didn't": "did not", "won't": "will not",
    "can't": "cannot", "couldn't": "could not", "wouldn't": "would not", "shouldn't": "should not",
    "isn't": "is not", "aren't": "are not", "wasn't": "was not", "weren't": "were not",
    "haven't": "have not", "hasn't": "has not", "hadn't": "had not", "mustn't": "must not",
    "i'm": "i am", "you're": "you are", "he's": "he is", "she's": "she is", "it's": "it is",
    "we're": "we are", "they're": "they are", "i've": "i have", "you've": "you have",
    "we've": "we have", "they've": "they have", "i'd": "i would", "you'd": "you would",
    "he'd": "he would", "she'd": "she would", "we'd": "we would", "they'd": "they would",
    "i'll": "i will", "you'll": "you will", "he'll": "he will", "she'll": "she will",
    "we'll": "we will", "they'll": "they will", "that's": "that is", "there's": "there is",
    "here's": "here is", "what's": "what is", "who's": "who is", "let's": "let us",
    "that'll": "that will", "there'll": "there will", "this'll": "this will",
}

def expand_contractions(text):
    t = text
    for cont, exp in CONTRACTIONS.items():
        t = re.sub(re.escape(cont), exp, t, flags=re.IGNORECASE)
    return t

# Load abbreviations except common/device tokens
def _load_abbrev_patterns():
    BLOCKED_KEYS = frozenset({
        # common words
        "so", "as", "or", "if", "is", "it", "in", "on", "at", "to", "of", "and",
        "the", "a", "an", "i", "you", "we", "they", "he", "she", "me", "my",
        "be", "do", "go", "no", "up", "by", "us",
        # device/tech
        "ios", "android", "iphone", "ipad", "macos", "usb", "usb-c", "lte", "nfc",
        "cpu", "gpu", "ram", "rom", "oled", "hdr", "heic", "jpeg", "raw", "av1",
        "api", "ui", "ux", "aosp", "oem", "arm", "tsmc",
    })

    ds = load_dataset("willwade/txt-sms-abbreviations", split="train")

    # Normalize key for abbreviation lookups by lowercasing and removing underscores
    def _norm_key(a):
        return a.lower().replace("_", "")

    # Avoids expanding blocked keys + device/acronym/model tokens
    def _is_protected_token(a, low_a):
        core = a.replace("_", "")
        if low_a in BLOCKED_KEYS:
            return True
        # Protect model tokens e.g., S23, 128GB
        if any(ch.isalpha() for ch in core) and any(ch.isdigit() for ch in core):
            return True
        # Protect common uppercase acronyms
        if 2 <= len(core) <= 6 and core.isupper():
            return True
        return False
    by_key = {}

    # Iterate through abbreviation dataset and build mapping for expansion
    for row in ds:
        a = str(row.get("Abbreviation", "") or "").strip()
        e = str(row.get("Expansion", "") or "").strip()
        if not a or not e:
            continue
        if len(a) == 1 and a.isdigit():
            continue
        low_a = _norm_key(a)
        if _is_protected_token(a, low_a):
            continue

        rep = f" {e} "
        # Take first occurrence of abbreviation for mapping
        if low_a not in by_key:
            by_key[low_a] = (a, rep)

    mapping = list(by_key.values())

    # Sort so that longer abbreviations match first to prevent substring issues
    mapping.sort(key=lambda x: -len(x[0]))

    # Construct regex patterns for each abbreviation
    patterns = []
    for a, rep in mapping:
        esc = re.escape(a)
        # Use word boundaries for alphanumeric abbreviations, otherwise, use custom boundaries
        pat = (r"\b" + esc + r"\b") if a.replace("_", "").isalnum() else (r"(?<!\w)" + esc + r"(?!\w)")
        patterns.append((pat, rep))
    return patterns

ABBREV_PATTERNS = _load_abbrev_patterns()

# Function to normalize microtext
def normalize_microtext(text):
    t = str(text or "")

    # Expand forms e.g., "2yrs" -> "2 years"
    def _years_repl(m):
        n = m.group(1)
        return f"{n} year" if n == "1" else f"{n} years"

    # Expand forms e.g., "3mos" -> "3 months"
    def _months_repl(m):
        n = m.group(1)
        return f"{n} month" if n == "1" else f"{n} months"

    t = re.sub(r"\b(\d+)\s*yr(s)?\b", _years_repl, t, flags=re.IGNORECASE)
    t = re.sub(r"\b(\d+)\s*mo(s)?\b", _months_repl, t, flags=re.IGNORECASE)

    for pat, rep in ABBREV_PATTERNS:
        t = re.sub(pat, rep, t, flags=re.IGNORECASE)

    # Reduce character elongation e.g., "sooo coool" -> "soo cool"
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    return t

# Replace emoticons with their corresponding words
EMOTICON_MAP = [
    (r":\)", " happy "), (r":\(", " sad "), (r":D", " happy "),
    (r">_<", " frustrated "), (r"<3", " love "),
]

def replace_emoticons(text):
    t = text
    for pat, rep in EMOTICON_MAP:
        t = re.sub(pat, rep, t)
    return t

# Replace emojis with their corresponding words
EMOJI_MAP = {
    "🆘": " distress ",
    "®": " registered ",
    "1⃣": " one ",
    "2⃣": " two "
}

def replace_emojis(text):
    t = text
    for em, rep in EMOJI_MAP.items():
        t = t.replace(em, rep)
    return t

def remove_urls(text):
    return re.sub(r"https?://\S+|www\.\S+", " ", text)

def remove_hashtags(text):
    return re.sub(r"#\w+", " ", text)

def remove_handles(text):
    return re.sub(r"@\w+", " ", text)

# Normalize percent expressions e.g., "10%" -> "10 percent"
_percent_re = re.compile(r"(?<!\\w)(\\d+(?:\\.\\d+)?)\\s*%(?!\\w)")

def normalize_percent(text):
    return _percent_re.sub(r"\\1 percent", text)

def correct_spelling(text):
  return str(TextBlob(text).correct())

# Remove special characters e.g., accents, punctuation
def remove_special_chars(text):
    t = unicodedata.normalize("NFKD", text)
    t = "".join(c for c in t if not unicodedata.combining(c))
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

# Lemmatize text
_lemmatizer = WordNetLemmatizer()

def _get_wordnet_pos(tag):
    if tag.startswith("J"): return wordnet.ADJ
    if tag.startswith("V"): return wordnet.VERB
    if tag.startswith("N"): return wordnet.NOUN
    if tag.startswith("R"): return wordnet.ADV
    return wordnet.NOUN

def lemmatize_text(text):
    tokens = word_tokenize(text)     # Tokenizes input text, tags each word with its POS
    tags = pos_tag(tokens)      # then lemmatizes each token using its POS tag
    return " ".join(_lemmatizer.lemmatize(w, _get_wordnet_pos(t)) for w, t in tags)

def normalize_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

def preprocess_for_traditional_ml(text, use_spell=True, use_lemma=True):
    t = str(text or "").strip()
    t = remove_hn_blockquotes(t)
    t = remove_markdown_formatting(t)
    t = remove_urls(t)
    t = remove_hashtags(t)
    t = remove_handles(t)
    t = replace_emoticons(t)
    t = replace_emojis(t)
    t = expand_contractions(t)
    t = normalize_microtext(t)
    t = normalize_percent(t)
    t = t.lower()
    t = remove_special_chars(t)
    if use_spell:
      t = correct_spelling(t)
    if use_lemma:
        t = lemmatize_text(t)
    return normalize_whitespace(t)

def preprocess_for_transformers(text):
    t = str(text or "").strip()
    t = remove_hn_blockquotes(t)
    t = remove_markdown_formatting(t)
    t = remove_urls(t)
    t = remove_hashtags(t)
    t = remove_handles(t)
    t = replace_emoticons(t)
    t = replace_emojis(t)
    t = expand_contractions(t)
    t = normalize_microtext(t)
    return normalize_whitespace(t)

## Apply preprocessing

In [38]:
USE_LEMMATIZATION = True
USE_SPELL_CORRECTION = False

df_clean = df.copy()
df_clean["text_ml"] = df_clean["text"].apply(
    lambda x: preprocess_for_traditional_ml(
        x, use_spell=USE_SPELL_CORRECTION, use_lemma=USE_LEMMATIZATION
    )
)
df_clean["text_ml_no_lemma"] = df_clean["text"].apply(
    lambda x: preprocess_for_traditional_ml(
        x, use_spell=USE_SPELL_CORRECTION, use_lemma=False
    )
)
df_clean["text_transformer"] = df_clean["text"].apply(preprocess_for_transformers)
print(f"Rows: {len(df_clean)}")

Rows: 50000


In [39]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 29 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   review_id             50000 non-null  int64  
 1   customer_name         50000 non-null  object 
 2   age                   50000 non-null  int64  
 3   brand                 50000 non-null  object 
 4   model                 50000 non-null  object 
 5   price_usd             50000 non-null  float64
 6   price_local           50000 non-null  object 
 7   currency              50000 non-null  object 
 8   exchange_rate_to_usd  50000 non-null  float64
 9   rating                50000 non-null  int64  
 10  review_text           50000 non-null  object 
 11  sentiment             50000 non-null  object 
 12  country               50000 non-null  object 
 13  language              50000 non-null  object 
 14  review_date           50000 non-null  object 
 15  verified_purchase  

In [40]:
text_col = "text"  
ml_col = "text_ml"
ml_no_lemma_col = "text_ml_no_lemma"
tr_col = "text_transformer"

cols = [c for c in [text_col, ml_col, ml_no_lemma_col, tr_col] if c in df_clean.columns]
df_clean[cols].sample(8, random_state=42)

,text,text_ml,text_ml_no_lemma,text_transformer
33553,"Does what it’s supposed to, nothing special. O...",do what it s suppose to nothing special okay f...,does what it s supposed to nothing special oka...,"Does what it’s supposed to, nothing special. O..."
9427,Software updates are delayed sometimes. Fine b...,software update be delay sometimes fine but co...,software updates are delayed sometimes fine bu...,Software updates are delayed sometimes. Fine b...
199,Build quality feels solid and durable. Absolut...,build quality feel solid and durable absolutel...,build quality feels solid and durable absolute...,Build quality feels solid and durable. Absolut...
12447,Sound quality is okay but not very loud. Fine ...,sound quality be okay but not very loud fine b...,sound quality is okay but not very loud fine b...,Sound quality is okay but not very loud. Fine ...
39489,Display is crisp and vibrant — perfect for gam...,display be crisp and vibrant perfect for game ...,display is crisp and vibrant perfect for gamin...,Display is crisp and vibrant — perfect for gam...
42724,Neutral feelings — neither great nor bad. Okay...,neutral feeling neither great nor bad okay for...,neutral feelings neither great nor bad okay fo...,Neutral feelings — neither great nor bad. Okay...
10822,Display is crisp and vibrant — perfect for gam...,display be crisp and vibrant perfect for game ...,display is crisp and vibrant perfect for gamin...,Display is crisp and vibrant — perfect for gam...
49498,Camera quality is very poor indoors. Not up to...,camera quality be very poor indoors not up to ...,camera quality is very poor indoors not up to ...,Camera quality is very poor indoors. Not up to...


In [41]:
for _, row in df_clean.sample(3, random_state=1).iterrows():
    print()
    print("Original:", row["text"])
    print("ML (lemma):", row["text_ml"])
    print("ML (no lemma):", row["text_ml_no_lemma"])
    print("Transformer:", row["text_transformer"])


Original: Absolutely love this phone! The camera is next level. Absolutely worth it!
ML (lemma): absolutely love this phone the camera be next level absolutely worth it
ML (no lemma): absolutely love this phone the camera is next level absolutely worth it
Transformer: Absolutely love this phone! The camera is next level. Absolutely worth it!

Original: Fast charging is a lifesaver. Best purchase of the year!
ML (lemma): fast charging be a lifesaver best purchase of the year
ML (no lemma): fast charging is a lifesaver best purchase of the year
Transformer: Fast charging is a lifesaver. Best purchase of the year!

Original: Not bad for daily use but could be optimized. Average experience overall.
ML (lemma): not bad for daily use but could be optimize average experience overall
ML (no lemma): not bad for daily use but could be optimized average experience overall
Transformer: Not bad for daily use but could be optimized. Average experience overall.


In [42]:
# Add subjectivity column
subjectivity_map = {
    'Positive': 'Subjective',
    'Negative': 'Subjective',
    'Neutral': 'Objective',
}

df_clean = df_clean.rename(columns={'sentiment': 'label_sentiment'})
df_clean['label_subjectivity'] = df_clean['label_sentiment'].map(subjectivity_map)

print(df_clean[['label_sentiment', 'label_subjectivity']].head(), "\n")
print(df_clean['label_sentiment'].value_counts(dropna=False), "\n")
print(df_clean['label_subjectivity'].value_counts(dropna=False))

  label_sentiment label_subjectivity
0        Negative         Subjective
1        Positive         Subjective
2        Positive         Subjective
3        Positive         Subjective
4         Neutral          Objective 

label_sentiment
Positive    27540
Neutral     12549
Negative     9911
Name: count, dtype: int64 

label_subjectivity
Subjective    37451
Objective     12549
Name: count, dtype: int64


In [43]:
# Save full dataset after preprocessing (Kaggle — does not overwrite HN corpus_preprocessed)
out_parquet = "../../data/preprocessed-data/corpus_training_kaggle.parquet"
df_clean.to_parquet(out_parquet, index=False)
print(f"Saved {out_parquet} | rows: {len(df_clean)} | cols: {len(df_clean.columns)}")

out_csv = "../../data/preprocessed-data/corpus_training_kaggle.csv"
df_clean.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"Saved {out_csv}")

Saved ../../data/preprocessed-data/corpus_training_kaggle.parquet | rows: 50000 | cols: 30
Saved ../../data/preprocessed-data/corpus_training_kaggle.csv
